# THE GOAT DEBATE — GOAT Calculator, Trend Explorer & Final Verdict
**Michael Jordan · Kobe Bryant · LeBron James**

| Section | What it does |
|---|---|
| **1 · GOAT Calculator** | Weight 15 stats and accolades on a 1–10 scale. For every player the table shows the career number, its normalized (0–100) score and the weighted score, and the totals and GOAT ranking update live. |
| **2 · Trend Explorer** | Pick a metric and watch the three careers side by side. The panel on the right turns that metric into a head-to-head scoreboard with (lightly) playful commentary. |
| **3 · Final Verdict** | Simulates every possible slider combination (exact counts on coarse grids, a 20-million-combination sample of the full 1–10 scale, and an exact lock-out test) and counts how many times each player is Rank #1. |

**Data sources (all in the repo):** `game_logs_summary_career_overall.csv` and `player_accolades.csv` (GOAT Calculator), `game_logs_summary_per_season.csv` (Trend Explorer; the accolades also mark title and MVP seasons on the trend chart).

**How to run:** *Run > Run All Cells* in JupyterLab. Needs `plotly` and `ipywidgets` (both in `requirements.txt`).

**Headshots (off by default):** the scorecards show only the rank and total. To add images, set `SHOW_HEADSHOTS = True` in the config cell and save pictures you have the rights to in `notebooks/assets/headshots/` named `michael_jordan`, `kobe_bryant` and `lebron_james` (`.png`, `.jpg`, `.jpeg` or `.webp`); a player without a file shows initials. Images are shown without a frame, so use ones with a transparent background or the card colour `#121A2B`; `HEADSHOT_STYLE = "circle"` gives round, ringed photos instead.

In [ ]:
# ── 0 · CONFIG, PALETTE & STYLE ───────────────────────────────────────────────
import base64
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import ipywidgets as W
from IPython.display import display, HTML, clear_output

warnings.filterwarnings("ignore", category=FutureWarning)

# --- data locations (first folder holding the file wins) -------------------------
DATA_DIR_CANDIDATES = [Path("../data/processed"), Path("data/processed"), Path("data"), Path(".")]
CAREER_FILE = "game_logs_summary_career_overall.csv"
SEASON_FILE = "game_logs_summary_per_season.csv"
ACC_FILE = "player_accolades.csv"
ACC_DIR_CANDIDATES = [Path("../data/external"), Path("data/external"), Path("data"), Path(".")]
ACC_AS_OF = "2026-11-05"
HEADSHOT_DIRS = [Path("assets/headshots"), Path("../assets/headshots"), Path("notebooks/assets/headshots"), Path("../notebooks/assets/headshots")]

# --- behaviour switches -----------------------------------------------------------
PLOTLY_RENDERER = "plotly_mimetype+notebook_connected"   # use "notebook" for fully offline use
MIN_GAMES_QUALIFIED = 30                                  # seasons below this are flagged "low-game"
WEIGHT_MIN, WEIGHT_MAX, WEIGHT_DEFAULT = 1, 10, 5         # calculator slider scale
SHOW_HEADSHOTS = False                                    # True: show a photo/avatar on each scorecard (see notebooks/assets/headshots)
HEADSHOT_STYLE = "blend"                                  # "blend": image sits directly on the card (no frame); "circle": round crop with a colour ring
pio.renderers.default = PLOTLY_RENDERER

# --- palette & helpers -----------------------------------------------------------
THEME = dict(
    bg="#0B0F1A", panel="#121A2B", panel_alt="#17223A", border="#1F2A44",
    text="#E8ECF4", muted="#8A96B0", grid="#1B2540",
    nba_blue="#1D428A", nba_blue_light="#3B6FD4", nba_red="#C8102E",
)


PLAYERS = ["Michael Jordan", "Kobe Bryant", "LeBron James"]


PLAYER_COLORS = {"Michael Jordan": "#CE1141",   # Bulls red
                 "Kobe Bryant":    "#8A5CC9",   # Lakers purple (lightened for dark UI)
                 "LeBron James":   "#FDBB30"}


PLAYER_SLUG = {"Michael Jordan": "mj", "Kobe Bryant": "kb", "LeBron James": "lj"}


SYMBOLS = {"Michael Jordan": "circle", "Kobe Bryant": "diamond", "LeBron James": "square"}


DASHES = {"Michael Jordan": "solid", "Kobe Bryant": "solid", "LeBron James": "solid"}   # all lines solid; players are told apart by colour and marker shape


NAME_FIX = {"Lebron James": "LeBron James"}


FONT = "Inter, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif"


PLOT_CONFIG = dict(displaylogo=False, modeBarButtonsToRemove=["lasso2d", "select2d"])


def rgba(hex_color, alpha):
    h = hex_color.lstrip("#")
    r, g, b = (int(h[i:i + 2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g},{b},{alpha})"


def style(fig, title=None, height=430, legend=True, margin=None):
    """Apply the dark NBA theme to a Plotly figure."""
    fig.update_layout(
        template="plotly_dark",
        paper_bgcolor=THEME["panel"], plot_bgcolor=THEME["panel"],
        font=dict(family=FONT, color=THEME["text"], size=12),
        title=dict(text=title, x=0.01, xanchor="left", font=dict(size=16)) if title else None,
        height=height,
        margin=margin or dict(l=60, r=30, t=70 if title else 40, b=55),
        showlegend=legend,
        legend=dict(orientation="h", x=1, xanchor="right", y=1.0, yanchor="bottom", bgcolor="rgba(0,0,0,0)"),
        hoverlabel=dict(bgcolor=THEME["bg"], bordercolor=THEME["border"], font=dict(color=THEME["text"])),
    )
    fig.update_xaxes(gridcolor=THEME["grid"], zerolinecolor=THEME["grid"], linecolor=THEME["border"])
    fig.update_yaxes(gridcolor=THEME["grid"], zerolinecolor=THEME["grid"], linecolor=THEME["border"])
    return fig


def show(fig):
    fig.show(config=PLOT_CONFIG)


def caption(text):
    return HTML(f'<div class="cap">{text}</div>')

PLAYER_TEXT = {"Michael Jordan": "#FFFFFF", "Kobe Bryant": "#FFFFFF", "LeBron James": "#1B1405"}   # readable text on each player colour
PLAYER_SHORT = {"Michael Jordan": "MJ", "Kobe Bryant": "Kobe", "LeBron James": "LeBron"}
PLAYER_INITIALS = {"Michael Jordan": "MJ", "Kobe Bryant": "KB", "LeBron James": "LJ"}

CSS = """
<style>
.goat-banner{background:linear-gradient(135deg,@panel 0%,@bg 100%);border:1px solid @border;border-radius:12px;
  padding:22px 26px 0;margin:6px 0 14px;overflow:hidden;font-family:@font}
.gb-title{font-size:30px;font-weight:800;letter-spacing:.06em;color:@text}
.gb-title span{color:@nba_red}
.gb-sub{color:@muted;font-size:14px;margin:4px 0 16px}
.gb-stripe{height:6px;margin:0 -26px;background:linear-gradient(90deg,@nba_red 0 33%,@text 33% 34%,@nba_blue 34% 100%)}
.cap{color:@muted;font-size:12.5px;margin:2px 4px 12px;font-family:@font;line-height:1.45}
.warnbox{background:#3a1018;border-left:4px solid @nba_red;border-radius:6px;padding:8px 14px;margin:6px 0;color:@text;font-size:13px}
.sec{color:@text;font-size:13px;letter-spacing:.12em;text-transform:uppercase;font-weight:700;margin:16px 0 6px;
  border-bottom:2px solid @nba_blue;padding-bottom:4px;font-family:@font}
.goat-root{background:@bg;border:1px solid @border;border-radius:12px;padding:14px}
.goat-root .widget-label,.goat-root .widget-readout,.goat-root label{color:@text !important}
.goat-root select,.goat-root input[type=text]{background:@panel;color:@text;border:1px solid @border}
.goat-root button.jupyter-button{background:@nba_blue;color:#fff;border:none}
.goat-root button.jupyter-button:hover{background:@nba_red}
.goat-root button.jupyter-button.mod-active{background:@nba_red}
.goat-root input[type=range]{accent-color:@nba_red}
.goat-root .widget-checkbox input{accent-color:@nba_red}
.goat-root .widget-html-content{width:100%;overflow:visible;line-height:normal}

/* ---- Trend Explorer: insights panel ---- */
.ins-wrap{background:@panel;border:1px solid @border;border-radius:12px;overflow:hidden;font-family:@font;color:@text}
.ins-head{padding:14px 16px 12px;border-bottom:3px solid @nba_red;background:linear-gradient(135deg,@panel_alt 0%,@panel 100%)}
.ins-kicker{font-size:11px;letter-spacing:.14em;text-transform:uppercase;color:@muted}
.ins-title{font-size:19px;font-weight:800;margin:3px 0 4px;line-height:1.25}
.ins-sub{font-size:12px;color:@muted;line-height:1.4}
.itbl{border-collapse:collapse;width:100%;font-size:12.8px}
.itbl td{vertical-align:top;padding:10px 12px;border-bottom:1px solid @border}
.itbl tr:last-child td{border-bottom:none}
.itbl td.ik{width:34%;background:@panel_alt;border-right:1px solid @border}
.itbl td.ik b{display:block;font-size:13px;margin-bottom:2px}
.itbl td.ik span.s{display:block;font-size:11px;color:@muted;line-height:1.3;margin-bottom:6px}
.chip{display:inline-block;font-size:11px;font-weight:800;padding:2px 8px;border-radius:10px;letter-spacing:.03em}
.itbl td.iv{line-height:1.5}
.ins-foot{padding:8px 14px;font-size:11px;color:@muted;border-top:1px solid @border}

/* ---- GOAT Calculator (fixed-width rows, so it renders identically in JupyterLab, Notebook 7 and VS Code) ---- */
.gc-title{font-size:26px;font-weight:800;letter-spacing:.04em;color:@text;font-family:@font;margin:2px 0}
.gc-title span{color:@nba_red}
.gc-lab{display:flex;flex-direction:column;justify-content:flex-end;font-family:@font;box-sizing:border-box;padding-bottom:4px}
.gc-lab div.t{height:34px;line-height:34px;padding-left:10px;color:@muted;font-size:12px;font-weight:700;letter-spacing:.07em;text-transform:uppercase}
.gc-card{background:@panel;border:1px solid @border;border-top:6px solid var(--c);text-align:center;padding:8px 4px 4px;font-family:@font;box-sizing:border-box}
.gc-card.lead{box-shadow:inset 0 0 0 2px var(--c)}
.gc-photo{width:110px;height:110px;border-radius:50%;margin:0 auto 6px;overflow:hidden;border:3px solid var(--c);box-sizing:border-box}
.gc-photo.blend{border:none;border-radius:0}
.gc-photo img{width:100%;height:100%;object-fit:cover;display:block}
.gc-photo.blend img{object-fit:contain}
.gc-init{width:100%;height:100%;display:flex;align-items:center;justify-content:center;font-size:30px;font-weight:800;letter-spacing:.04em}
.gc-rank{height:34px;line-height:34px;font-size:26px;font-weight:800;color:var(--c)}
.gc-score{height:34px;line-height:34px;font-size:19px;font-weight:700;color:@text;font-variant-numeric:tabular-nums}
.gc-name{font-family:@font;font-weight:800;font-size:14px;text-align:center;height:32px;line-height:32px;box-sizing:border-box;letter-spacing:.02em;white-space:nowrap}
.gc-hd{font-family:@font;background:#05070D;color:#fff;font-weight:700;font-size:12px;padding:0 8px;height:44px;display:flex;align-items:center;border-bottom:3px solid @nba_red;box-sizing:border-box}
.gc-sh{font-family:@font;color:@text;font-weight:700;font-size:11.5px;line-height:1.15;text-align:center;padding:0 3px;height:44px;display:flex;align-items:center;justify-content:center;box-sizing:border-box}
.gc-sec{font-family:@font;color:@muted;font-size:10.5px;letter-spacing:.14em;text-transform:uppercase;height:28px;display:flex;align-items:flex-end;padding:0 10px 4px;border-bottom:2px solid @nba_blue;box-sizing:border-box}
.gc-m{font-family:@font;padding:0 8px;color:@text;background:@panel;border-bottom:1px solid @border;font-size:12.5px;line-height:1.15;height:36px;display:flex;flex-direction:column;justify-content:center;box-sizing:border-box;white-space:nowrap}
.gc-m.acc{color:#9DBBFF}
.gc-m i{color:@muted;font-style:normal;font-size:10px}
.gc-v{font-family:@font;text-align:center;font-variant-numeric:tabular-nums;color:@text;background:@panel;border-bottom:1px solid @border;border-right:1px solid @border;font-size:13px;height:36px;display:flex;align-items:center;justify-content:center;box-sizing:border-box}
.gc-v.w{background:@panel_alt;color:#C9D3EA}
.gc-tot-lab{font-family:@font;color:@text;font-weight:800;font-size:12.5px;letter-spacing:.08em;text-transform:uppercase;padding:0 10px;background:#05070D;border-top:3px solid @nba_red;height:44px;display:flex;align-items:center;box-sizing:border-box}
.gc-tot{font-family:@font;font-weight:800;font-size:21px;background:@panel;border-top:3px solid @nba_red;font-variant-numeric:tabular-nums;height:44px;display:flex;align-items:center;justify-content:center;box-sizing:border-box}
.gc-wcell{background:@panel;border-bottom:1px solid @border;padding:0 8px;box-sizing:border-box}
.gc-share{font-family:@font;color:@muted;font-size:11px;white-space:nowrap}
.gc-share b{color:@text}
.gdesc{background:@panel;border:1px solid @border;border-left:4px solid @nba_blue_light;border-radius:10px;padding:14px 20px;color:@text;
  font-family:@font;font-size:13px;line-height:1.6;margin-top:12px;box-sizing:border-box}
.gdesc h4{margin:0 0 6px;font-size:13.5px;letter-spacing:.05em;text-transform:uppercase}
.gdesc ol{margin:6px 0 8px;padding-left:20px}
.gdesc .ex{background:@panel_alt;border-radius:8px;padding:8px 12px;margin-top:8px;font-size:12.5px}
/* ---- Final Verdict ---- */
.fvt{border-collapse:collapse;width:100%;font-size:13px;font-family:@font;color:@text}
.fvt th{background:#05070D;padding:9px 10px;text-align:center;font-size:12px;border-bottom:3px solid @nba_red;vertical-align:middle}
.fvt th.l,.fvt td.l{text-align:left}
.fvt td{padding:9px 10px;border-bottom:1px solid @border;text-align:center;background:@panel;font-variant-numeric:tabular-nums;vertical-align:middle}
.fvt.tight td{padding:5px 10px}
.fvt .sub{display:block;color:@muted;font-size:11px;font-weight:400;margin-top:1px}
__PLAYER_CSS__
</style>
"""
_player_css = "\n".join(f".pbox-{PLAYER_SLUG[p]} label{{color:{c} !important;font-weight:700}}" for p, c in PLAYER_COLORS.items())
CSS = CSS.replace("__PLAYER_CSS__", _player_css).replace("@font", FONT)
for _k in sorted(THEME, key=len, reverse=True):
    CSS = CSS.replace("@" + _k, THEME[_k])
display(HTML(CSS))


def banner_html(title_html, subtitle):
    return (f'<div class="goat-banner"><div class="gb-title">{title_html}</div>'
            f'<div class="gb-sub">{subtitle}</div><div class="gb-stripe"></div></div>')


display(HTML(banner_html('THE <span>GOAT</span> DEBATE',
                         'Michael Jordan vs Kobe Bryant vs LeBron James · GOAT calculator, trend explorer &amp; final verdict')))

## Data: load & validate
Three sources: the per-season and career-overall game-log summaries, plus the hand-compiled accolades table.
* Per-season points are not in the file, so they are derived as `2·FGM + 3PM + FTM` and reconciled with the career `PTS_AVE`.
* Shooting totals and per-game averages combine regular season **and** playoffs.
* Accolades (championships, awards, honours) were compiled on 5 Nov 2026 from NBA.com, ESPN and Wikipedia/Basketball-Reference-based lists. Edit `data/external/player_accolades.csv` to correct or extend them.

In [ ]:
def find_data_dir():
    for d in DATA_DIR_CANDIDATES:
        if (d / CAREER_FILE).exists() and (d / SEASON_FILE).exists():
            return d
    raise FileNotFoundError(
        f"Could not find {CAREER_FILE} and {SEASON_FILE} in any of: "
        + ", ".join(str(p) for p in DATA_DIR_CANDIDATES)
        + ". Edit DATA_DIR_CANDIDATES in the config cell.")


def add_season_features(df):
    d = df.copy()
    gp = d["GAMES_PLAYED_TOTAL"]
    d["SEASON_START"] = d["SEASON_YEAR"].str[:4].astype(int)
    d["PTS_TOTAL"] = 2 * d["TOTAL_FGM"] + d["TOTAL_FG3M"] + d["TOTAL_FTM"]
    d["PPG"] = d["PTS_TOTAL"] / gp
    d["EFG"] = (d["TOTAL_FGM"] + 0.5 * d["TOTAL_FG3M"]) / d["TOTAL_FGA"]
    d["TS"] = d["PTS_TOTAL"] / (2 * (d["TOTAL_FGA"] + 0.44 * d["TOTAL_FTA"]))
    d["FG3A_PG"] = d["TOTAL_FG3A"] / gp
    d["FTA_PG"] = d["TOTAL_FTA"] / gp
    d["FG3A_RATE"] = d["TOTAL_FG3A"] / d["TOTAL_FGA"]
    d["FT_RATE"] = d["TOTAL_FTA"] / d["TOTAL_FGA"]
    d["PLAYOFF_SHARE"] = d["GAMES_PLAYED_PLAYOFFS"] / gp
    d["STOCKS"] = d["STL_AVE"] + d["BLK_AVE"]
    d["AST_TOV"] = d["AST_AVE"] / d["TOV_AVE"]
    d["QUALIFIED"] = gp >= MIN_GAMES_QUALIFIED
    d["LOW_GAMES"] = ~d["QUALIFIED"]
    return d.sort_values(["PLAYER_NAME", "PLAYER_SEASON_NUM"]).reset_index(drop=True)


DATA_DIR = find_data_dir()
CAREER = pd.read_csv(DATA_DIR / CAREER_FILE)
SEASON = pd.read_csv(DATA_DIR / SEASON_FILE)
for _df in (CAREER, SEASON):
    _df["PLAYER_NAME"] = _df["PLAYER_NAME"].str.strip().replace(NAME_FIX)
SEASON = add_season_features(SEASON)
SEASON_BY_PLAYER = {p: g.sort_values("PLAYER_SEASON_NUM").reset_index(drop=True) for p, g in SEASON.groupby("PLAYER_NAME")}
CAREER_X = CAREER.set_index("PLAYER_NAME").loc[PLAYERS]

# --- reconcile the two game-log files ---------------------------------------------
_agg = SEASON.groupby("PLAYER_NAME").agg(pts=("PTS_TOTAL", "sum"), gp=("GAMES_PLAYED_TOTAL", "sum"), seasons=("SEASON_YEAR", "count"))
for p in PLAYERS:
    c = CAREER_X.loc[p]
    assert abs(_agg.loc[p, "pts"] / _agg.loc[p, "gp"] - c["PTS_AVE"]) < 0.01, f"PPG mismatch for {p}"
    assert _agg.loc[p, "gp"] == c["GAMES_PLAYED_TOTAL"], f"Games mismatch for {p}"
    assert _agg.loc[p, "seasons"] == c["TOTAL_SEASONS_PLAYED"], f"Season count mismatch for {p}"
assert set(PLAYERS) == set(SEASON["PLAYER_NAME"]) == set(CAREER["PLAYER_NAME"]), "Unexpected player names"
_need = ["TOTAL_FG_PCT", "TOTAL_FG3_PCT", "TOTAL_FT_PCT", "PTS_AVE", "REB_AVE", "AST_AVE", "STL_AVE", "BLK_AVE", "TOV_AVE"]
assert all(c in CAREER.columns for c in _need), "Career file is missing a column the calculator needs"

# --- accolades (external, editable) -----------------------------------------------
_ACC_EMBEDDED = {
    "Michael Jordan": dict(championships=6, finals_appearances=6, finals_mvp=6, mvp=5, dpoy=1, scoring_titles=10, allstar_mvp=3,
                           allstar_selections=14, allnba_first=10, allnba_second=1, allnba_third=0, alldefensive=9, olympic_gold=2,
                           champion_seasons="1990-91;1991-92;1992-93;1995-96;1996-97;1997-98",
                           mvp_seasons="1987-88;1990-91;1991-92;1995-96;1997-98"),
    "Kobe Bryant":    dict(championships=5, finals_appearances=7, finals_mvp=2, mvp=1, dpoy=0, scoring_titles=2, allstar_mvp=4,
                           allstar_selections=18, allnba_first=11, allnba_second=2, allnba_third=2, alldefensive=12, olympic_gold=2,
                           champion_seasons="1999-00;2000-01;2001-02;2008-09;2009-10",
                           mvp_seasons="2007-08"),
    "LeBron James":   dict(championships=4, finals_appearances=10, finals_mvp=4, mvp=4, dpoy=0, scoring_titles=1, allstar_mvp=3,
                           allstar_selections=22, allnba_first=13, allnba_second=4, allnba_third=4, alldefensive=6, olympic_gold=3,
                           champion_seasons="2011-12;2012-13;2015-16;2019-20",
                           mvp_seasons="2008-09;2009-10;2011-12;2012-13"),
}


ACC_NUM_COLS = ["championships", "finals_appearances", "finals_mvp", "mvp", "dpoy", "scoring_titles", "allstar_mvp",
                "allstar_selections", "allnba_first", "allnba_second", "allnba_third", "alldefensive", "olympic_gold"]


def load_accolades():
    df, src = pd.DataFrame(_ACC_EMBEDDED).T, "table embedded in this notebook"
    for d in ACC_DIR_CANDIDATES:
        f = d / ACC_FILE
        if f.exists():
            df = pd.read_csv(f)
            df["PLAYER_NAME"] = df["PLAYER_NAME"].str.strip().replace(NAME_FIX)
            df, src = df.set_index("PLAYER_NAME"), str(f.resolve())
            break
    missing = [c for c in ACC_NUM_COLS + ["champion_seasons", "mvp_seasons"] if c not in df.columns]
    assert not missing, f"Accolades table is missing columns: {missing}"
    df = df.loc[PLAYERS].copy()
    df[ACC_NUM_COLS] = df[ACC_NUM_COLS].astype(int)
    return df, src


def season_list(p, col):
    return [x for x in str(ACC.loc[p, col]).split(";") if x and x != "nan"]


ACC, ACC_SOURCE = load_accolades()
ACC["allnba_total"] = ACC["allnba_first"] + ACC["allnba_second"] + ACC["allnba_third"]
for p in PLAYERS:
    a = ACC.loc[p]
    assert a.championships <= a.finals_appearances, f"{p}: titles exceed Finals appearances"
    assert len(season_list(p, "champion_seasons")) == a.championships, f"{p}: champion_seasons length != championships"
    assert len(season_list(p, "mvp_seasons")) == a.mvp, f"{p}: mvp_seasons length != mvp"
    known = set(SEASON_BY_PLAYER[p]["SEASON_YEAR"])
    assert set(season_list(p, "champion_seasons") + season_list(p, "mvp_seasons")) <= known, f"{p}: award season not in season data"

print(f"Game logs: {DATA_DIR.resolve()}\nAccolades: {ACC_SOURCE} (as of {ACC_AS_OF})\nValidation passed: files reconcile and accolade checks are consistent.\n")
display(pd.DataFrame({"Seasons": [int(_agg.loc[p, "seasons"]) for p in PLAYERS],
                      "First": [SEASON_BY_PLAYER[p]["SEASON_YEAR"].iloc[0] for p in PLAYERS],
                      "Last": [SEASON_BY_PLAYER[p]["SEASON_YEAR"].iloc[-1] for p in PLAYERS],
                      "Games (RS+PO)": [int(_agg.loc[p, "gp"]) for p in PLAYERS]}, index=PLAYERS))

# 1 · GOAT Calculator
Set a weight from 1 to 10 for each stat and accolade. For every player the table shows three numbers per row:
**Career** (the actual figure), **Normalized** (0–100, relative to the best of the three) and **Weighted Score** (normalized score × the row's share of the total weight). The totals, GOAT ranking and the "how it works" example below update live as you drag.
Numbers come from `game_logs_summary_career_overall.csv` (stats) and `player_accolades.csv` (accolades).

In [ ]:
CALC_ROWS = [   # (key, source, friendly name, display kind, higher_is_better)
    ("TOTAL_FG_PCT",  "career", "Field Goal %",             "pct",  True),
    ("TOTAL_FG3_PCT", "career", "3-Point %",                "pct",  True),
    ("TOTAL_FT_PCT",  "career", "Free Throw %",             "pct",  True),
    ("PTS_AVE",       "career", "Points per Game",          "num1", True),
    ("REB_AVE",       "career", "Rebounds per Game",        "num1", True),
    ("AST_AVE",       "career", "Assists per Game",         "num1", True),
    ("STL_AVE",       "career", "Steals per Game",          "num2", True),
    ("BLK_AVE",       "career", "Blocks per Game",          "num2", True),
    ("TOV_AVE",       "career", "Turnovers per Game",       "num1", False),
    ("championships", "acc",    "NBA Championships",        "int",  True),
    ("finals_mvp",    "acc",    "Finals MVP Awards",        "int",  True),
    ("mvp",           "acc",    "League MVP Awards",        "int",  True),
    ("dpoy",          "acc",    "Defensive Player of the Year", "int", True),
    ("scoring_titles", "acc",   "Scoring Titles",           "int",  True),
    ("allnba_first",  "acc",    "All-NBA 1st Team Selections", "int", True),
]
CALC_KEYS = [r[0] for r in CALC_ROWS]
N_STATS = sum(1 for r in CALC_ROWS if r[1] == "career")
assert len(set(CALC_KEYS)) == len(CALC_KEYS)

RAW = pd.DataFrame({k: [float(CAREER_X.loc[p, k] if src == "career" else ACC.loc[p, k]) for p in PLAYERS] for k, src, *_ in CALC_ROWS}, index=PLAYERS)


def _normalise(s, higher_better):
    s = s.astype(float)
    if higher_better:
        return (s / s.max() if s.max() > 0 else s * 0 + 1) * 100          # nobody has any -> everyone ties at 100
    return (s.min() / s) * 100                                              # fewer is better: best / value


NORM = pd.DataFrame({k: _normalise(RAW[k], hb) for k, _, _, _, hb in CALC_ROWS})
BEST = {k: [p for p in PLAYERS if NORM.loc[p, k] >= 100 - 1e-9] for k in CALC_KEYS}


def calc_fmt(v, kind):
    if kind == "pct":
        return f"{v * 100:.1f}%"
    if kind == "num1":
        return f"{v:.1f}"
    if kind == "num2":
        return f"{v:.2f}"
    return f"{int(round(v))}"


def calc_scores(weights):
    """weights: {key: 1-10}. Returns (share of total weight per key, weighted-score table, total per player, rank per player)."""
    tw = float(sum(weights[k] for k in CALC_KEYS))
    share = {k: weights[k] / tw for k in CALC_KEYS}
    wsc = pd.DataFrame({k: NORM[k] * share[k] for k in CALC_KEYS})
    total = wsc.sum(axis=1)
    rank = total.rank(ascending=False, method="min").astype(int)
    return share, wsc, total, rank


def headshot_html(p):
    slug = p.lower().replace(" ", "_")
    for d in HEADSHOT_DIRS:
        for ext, mime in (("png", "image/png"), ("jpg", "image/jpeg"), ("jpeg", "image/jpeg"), ("webp", "image/webp")):
            f = d / f"{slug}.{ext}"
            if f.exists():
                b64 = base64.b64encode(f.read_bytes()).decode()
                cls = "gc-photo blend" if HEADSHOT_STYLE == "blend" else "gc-photo"
                return f'<div class="{cls}"><img src="data:{mime};base64,{b64}" alt="{p}"></div>'
    c = PLAYER_COLORS[p]
    return (f'<div class="gc-photo"><div class="gc-init" style="background:{rgba(c, 0.22)};color:{c}">{PLAYER_INITIALS[p]}</div></div>')


def _best_style(p, pos):
    c = PLAYER_COLORS[p]
    sides = {"l": f"inset 2px 0 0 {c},inset 0 2px 0 {c},inset 0 -2px 0 {c}",
             "m": f"inset 0 2px 0 {c},inset 0 -2px 0 {c}",
             "r": f"inset -2px 0 0 {c},inset 0 2px 0 {c},inset 0 -2px 0 {c}"}[pos]
    return f"box-shadow:{sides};background:{rgba(c, 0.15)};color:#FFFFFF;font-weight:700;"


def value_cell(p, k, kind, pos, text):
    cls = {"l": "gc-v", "m": "gc-v n", "r": "gc-v w"}[pos]
    style_ = _best_style(p, pos) if p in BEST[k] else ""
    return f'<div class="{cls}" style="{style_}">{text}</div>'


# ---- fixed layout (pixels): every row is built from the same column widths, so scorecards sit exactly above their player's 3 columns
COL_METRIC, COL_WEIGHT = 210, 353                       # Metric column unchanged; slider column is 1.5x the previous 235px
COL_SUB = (93, 105, 99)                                  # Career | Normalized | Weighted Score (1.5x the previous 62 / 70 / 66 px)
PLAYER_W = sum(COL_SUB)
LEFT_W = COL_METRIC + COL_WEIGHT
TABLE_W = LEFT_W + len(PLAYERS) * PLAYER_W
PHOTO_BLOCK = (110 + 6) if SHOW_HEADSHOTS else 0
CARD_H = 6 + 8 + PHOTO_BLOCK + 68 + 4                    # top border + padding + photo + rank/score rows + padding


def card_html(p, rank, total):
    star = " ★" if rank == 1 else ""
    photo = HEADSHOTS[p] if SHOW_HEADSHOTS else ""
    return (f'<div class="gc-card{" lead" if rank == 1 else ""}" style="--c:{PLAYER_COLORS[p]};height:{CARD_H}px">{photo}'
            f'<div class="gc-rank">{rank}{star}</div><div class="gc-score">{total:.2f}</div></div>')


def total_html(p, rank, total):
    c = PLAYER_COLORS[p]
    ring = f"box-shadow:inset 0 0 0 2px {c};background:{rgba(c, 0.15)};" if rank == 1 else ""
    return f'<div class="gc-tot" style="color:{c};{ring}">{total:.2f}</div>'


def desc_html(weights, share, wsc):
    k = "PTS_AVE"
    order = list(RAW[k].sort_values(ascending=False).index)
    best, p = order[0], order[1]
    ex = (f"<b>Worked example · Points per Game.</b> {PLAYER_SHORT[p]} averages {calc_fmt(RAW.loc[p, k], 'num1')} and the best in the group "
          f"({PLAYER_SHORT[best]}) averages {calc_fmt(RAW.loc[best, k], 'num1')}. Normalized = {calc_fmt(RAW.loc[p, k], 'num1')} ÷ {calc_fmt(RAW.loc[best, k], 'num1')} × 100 = "
          f"<b>{NORM.loc[p, k]:.1f}</b>. Your weight for this row is {weights[k]} out of {sum(weights.values())} in total, a share of <b>{share[k] * 100:.1f}%</b>. "
          f"Weighted score = {NORM.loc[p, k]:.1f} × {share[k] * 100:.1f}% = <b>{wsc.loc[p, k]:.2f}</b>.")
    return (
        '<div class="gdesc"><h4>How the weighted scores are normalized and calculated</h4><ol>'
        '<li><b>Career</b> is the real number. Stats are career per-game averages and shooting percentages (regular season + playoffs) from the career game-log summary; accolades are counts from the accolades table.</li>'
        '<li><b>Normalized (0–100)</b> compares each player with the best of the three in that row: <i>value ÷ best × 100</i>, so the leader always scores 100. Turnovers is flipped (<i>best ÷ value</i>) because fewer is better.</li>'
        f'<li><b>Share of the weight</b>: each slider (1–10) becomes <i>your weight ÷ the sum of all {len(CALC_KEYS)} weights</i>, so the shares always add up to 100%.</li>'
        '<li><b>Weighted Score</b> = <i>Normalized × share</i>: the points that row adds to the player\'s total.</li>'
        '<li><b>Total Weighted Score</b> is the sum of a player\'s weighted scores (a perfect 100 would mean leading every row). The highest total is GOAT #1.</li></ol>'
        f'<div class="ex">{ex}</div>'
        f'<div class="cap" style="margin-top:8px">An outlined row marks the best performer in that row (lowest for turnovers). Career-volume totals such as total points, total games and seasons played are deliberately not used, '
        f'so a longer career earns no automatic edge. Accolades were compiled on {ACC_AS_OF}.</div></div>')


HEADSHOTS = {p: headshot_html(p) for p in PLAYERS} if SHOW_HEADSHOTS else {}


def build_goat_calculator():
    sliders = {k: W.IntSlider(value=WEIGHT_DEFAULT, min=WEIGHT_MIN, max=WEIGHT_MAX, step=1, continuous_update=True, readout=True,
                              layout=W.Layout(width="177px", flex="0 0 auto")) for k in CALC_KEYS}
    share_lbl = {k: W.HTML(layout=W.Layout(width="138px", min_width="138px", flex="0 0 auto", margin="0 0 0 6px")) for k in CALC_KEYS}
    reset_btn = W.Button(description="Reset weights", icon="refresh", layout=W.Layout(width="150px"))

    def cell(html, width):
        return W.HTML(html, layout=W.Layout(width=f"{width}px", min_width=f"{width}px", flex="0 0 auto", margin="0"))

    cards = {p: cell("", PLAYER_W) for p in PLAYERS}
    totals = {p: cell("", PLAYER_W) for p in PLAYERS}
    wcells = {(p, k): cell("", COL_SUB[2]) for p in PLAYERS for k in CALC_KEYS}
    desc = W.HTML(layout=W.Layout(width=f"{TABLE_W}px"))
    kind_of = {r[0]: r[3] for r in CALC_ROWS}

    def row(children):
        return W.HBox(children, layout=W.Layout(width=f"{TABLE_W}px", min_width=f"{TABLE_W}px", flex_flow="row nowrap", align_items="stretch", margin="0"))

    rows = []
    # 1 · scorecards: label on the left, one card above each player's three columns
    rows.append(row([cell(f'<div class="gc-lab" style="height:{CARD_H}px"><div class="t">GOAT Ranking</div><div class="t">Total Weighted Score</div></div>', LEFT_W)]
                    + [cards[p] for p in PLAYERS]))
    # 2 · player name bands
    rows.append(row([cell("", LEFT_W)] + [cell(f'<div class="gc-name" style="background:{PLAYER_COLORS[p]};color:{PLAYER_TEXT[p]}">{p}</div>', PLAYER_W) for p in PLAYERS]))
    # 3 · column headers (they wrap to two lines when needed)
    hdr = [cell('<div class="gc-hd">Metric</div>', COL_METRIC), cell('<div class="gc-hd">Metric Weight (1–10)</div>', COL_WEIGHT)]
    for p in PLAYERS:
        c = PLAYER_COLORS[p]
        for txt, w in zip(("Career", "Normalized", "Weighted Score"), COL_SUB):
            hdr.append(cell(f'<div class="gc-sh" style="background:{rgba(c, 0.22)};border-bottom:3px solid {c}">{txt}</div>', w))
    rows.append(row(hdr))

    def data_row(k, src, name, kind, hb):
        acc = src == "acc"
        label = f'{name}{" †" if acc else ""}' + ("" if hb else "<i>↓ fewer is better</i>")
        box = W.HBox([sliders[k], share_lbl[k]], layout=W.Layout(width=f"{COL_WEIGHT}px", min_width=f"{COL_WEIGHT}px", height="36px", flex="0 0 auto",
                                                               align_items="center", margin="0"))
        box.add_class("gc-wcell")
        cs = [cell(f'<div class="gc-m{" acc" if acc else ""}">{label}</div>', COL_METRIC), box]
        for p in PLAYERS:
            cs.append(cell(value_cell(p, k, kind, "l", calc_fmt(RAW.loc[p, k], kind)), COL_SUB[0]))
            cs.append(cell(value_cell(p, k, kind, "m", f"{NORM.loc[p, k]:.1f}"), COL_SUB[1]))
            cs.append(wcells[(p, k)])
        return row(cs)

    rows.append(row([cell('<div class="gc-sec">Career stats · from game_logs_summary_career_overall</div>', TABLE_W)]))
    rows += [data_row(*r) for r in CALC_ROWS if r[1] == "career"]
    rows.append(row([cell('<div class="gc-sec">Accolades · from player_accolades †</div>', TABLE_W)]))
    rows += [data_row(*r) for r in CALC_ROWS if r[1] == "acc"]
    rows.append(row([cell('<div class="gc-tot-lab">Total Weighted Score</div>', LEFT_W)] + [totals[p] for p in PLAYERS]))
    table = W.VBox(rows, layout=W.Layout(width=f"{TABLE_W}px", min_width=f"{TABLE_W}px", margin="0"))

    def refresh(_=None):
        weights = {k: s_.value for k, s_ in sliders.items()}
        share, wsc, total, rank = calc_scores(weights)
        for k in CALC_KEYS:
            share_lbl[k].value = f'<span class="gc-share"><b>{share[k] * 100:.1f}%</b> of score</span>'
        for (p, k), wdg in wcells.items():
            wdg.value = value_cell(p, k, kind_of[k], "r", f"{wsc.loc[p, k]:.2f}")
        for p in PLAYERS:
            cards[p].value = card_html(p, int(rank[p]), float(total[p]))
            totals[p].value = total_html(p, int(rank[p]), float(total[p]))
        desc.value = desc_html(weights, share, wsc)

    def reset(_b=None):
        for s_ in sliders.values():
            s_.value = WEIGHT_DEFAULT

    for s_ in sliders.values():
        s_.observe(refresh, names="value")
    reset_btn.on_click(reset)

    root = W.VBox([
        W.HTML('<div class="gc-title">NBA <span>GOAT</span> Calculator</div>'),
        W.HTML(f'<div class="cap" style="max-width:{TABLE_W}px">Drag a slider ({WEIGHT_MIN} = barely matters, {WEIGHT_MAX} = matters most). Each player gets three numbers per row: '
               'the <b>Career</b> figure, its <b>Normalized</b> 0–100 score, and the <b>Weighted Score</b> it adds to the total. Outlines mark the best performer in each row.</div>'),
        reset_btn, table, desc,
    ], layout=W.Layout(overflow_x="auto"))
    root.add_class("goat-root")
    root.refs = dict(sliders=sliders, reset=reset_btn, cards=cards, totals=totals, wcells=wcells, desc=desc, table=table, rows=rows, refresh=refresh, reset_fn=reset)
    refresh()
    return root


GOAT_UI = build_goat_calculator()
display(GOAT_UI)

# 2 · Trend Explorer
Pick a metric and compare the three careers season by season. Use **career year** (rookies line up at year 1) or **calendar season** on the X-axis, smooth with a 3-season rolling average, and toggle the title ▲ / MVP ○ markers.
The panel on the right turns the chosen metric into a head-to-head scoreboard, with a little attitude. It uses qualifying seasons only (30+ games) and re-computes whenever you change the metric or the players.
Chart conventions: ★ = best season for the metric · hollow marker = low-game season · broken line = calendar gap in the data (e.g. retirement).

In [ ]:
def with_gaps(d, xcol):
    """Insert a NaN row wherever consecutive rows skip a calendar season, so lines break at retirements/missing years."""
    d = d.sort_values("PLAYER_SEASON_NUM").reset_index(drop=True)
    rows = []
    for i, row in d.iterrows():
        if i > 0 and row["SEASON_START"] - d.loc[i - 1, "SEASON_START"] > 1:
            blank = {c: np.nan for c in d.columns}
            blank[xcol] = (d.loc[i - 1, xcol] + row[xcol]) / 2
            rows.append(blank)
        rows.append(row.to_dict())
    return pd.DataFrame(rows)


METRICS = {
    "Points per game":              dict(col="PPG",                         kind="num", dec=1, hb=True),
    "True shooting % (TS%)":        dict(col="TS",                          kind="pct", dec=1, hb=True),
    "Effective FG% (eFG%)":         dict(col="EFG",                         kind="pct", dec=1, hb=True),
    "Field goal %":                 dict(col="TOTAL_FG_PCT",                kind="pct", dec=1, hb=True),
    "3-point %":                    dict(col="TOTAL_FG3_PCT",               kind="pct", dec=1, hb=True),
    "Free throw %":                 dict(col="TOTAL_FT_PCT",                kind="pct", dec=1, hb=True),
    "Assists per game":             dict(col="AST_AVE",                     kind="num", dec=1, hb=True),
    "Rebounds per game":            dict(col="REB_AVE",                     kind="num", dec=1, hb=True),
    "Steals per game":              dict(col="STL_AVE",                     kind="num", dec=2, hb=True),
    "Blocks per game":              dict(col="BLK_AVE",                     kind="num", dec=2, hb=True),
    "Turnovers per game":           dict(col="TOV_AVE",                     kind="num", dec=1, hb=False),
    "3-pt attempts per game":       dict(col="FG3A_PG",                     kind="num", dec=1, hb=True),
    "Free-throw attempts per game": dict(col="FTA_PG",                      kind="num", dec=1, hb=True),
    "Games played (regular season)": dict(col="GAMES_PLAYED_REGULAR_SEASON", kind="num", dec=0, hb=True),
    "Regular-season win %":         dict(col="REGULAR_SEASON_WIN_PCT",      kind="pct", dec=1, hb=True),
}


def fmt_val(v, m):
    if pd.isna(v):
        return "–"
    return f"{v * 100:.1f}%" if m["kind"] == "pct" else f"{v:,.{m['dec']}f}"


def hover_fmt(m):
    return ".1%" if m["kind"] == "pct" else f".{m['dec']}f"


def xinfo(xmode):
    if xmode == "Career year":
        return "PLAYER_SEASON_NUM", "Career year (1 = rookie season)"
    return "SEASON_START", "Season start year"


def add_series(fig, p, mkey, xcol, smooth, pos=None, legend=True, star=True, label=True, size=8, awards=False):
    """One player's line for one metric: gap-aware, hollow markers for low-game seasons, star on the career best."""
    m = METRICS[mkey]
    c = m["col"]
    d = SEASON_BY_PLAYER[p].copy()
    d["_RAW"] = d[c]
    d["_Y"] = d[c].rolling(3, center=True, min_periods=1).mean() if smooth else d[c]
    g = with_gaps(d, xcol)
    color = PLAYER_COLORS[p]
    low = g["LOW_GAMES"].eq(True).tolist()
    kw = dict(row=pos[0], col=pos[1]) if pos else {}
    hf = hover_fmt(m)
    cd = g[["SEASON_YEAR", "PLAYER_SEASON_NUM", "GAMES_PLAYED_TOTAL", "_RAW"]].astype(object).values.tolist()
    ht = (f"<b>{p}</b> · %{{customdata[0]}}<br>Career year %{{customdata[1]:.0f}} · games %{{customdata[2]:.0f}}"
          f"<br>{mkey}: %{{y:{hf}}}" + (f"<br>Raw value: %{{customdata[3]:{hf}}}" if smooth else "") + "<extra></extra>")
    if smooth:
        fig.add_trace(go.Scatter(x=g[xcol], y=g["_RAW"], mode="lines", connectgaps=False, hoverinfo="skip",
                                 line=dict(color=color, width=1), opacity=0.3, showlegend=False,
                                 legendgroup=p), **kw)
    fig.add_trace(go.Scatter(
        x=g[xcol], y=g["_Y"], mode="lines+markers", name=p, legendgroup=p, showlegend=legend, connectgaps=False,
        line=dict(color=color, width=2.5, dash=DASHES[p]),
        marker=dict(symbol=SYMBOLS[p], size=size, color=[THEME["panel"] if lw else color for lw in low],
                    line=dict(color=color, width=2)),
        customdata=cd, hovertemplate=ht), **kw)
    if star:
        q = d[d["QUALIFIED"]]
        idx = q["_Y"].idxmax() if m["hb"] else q["_Y"].idxmin()
        fig.add_trace(go.Scatter(
            x=[d.loc[idx, xcol]], y=[d.loc[idx, "_Y"]], mode="markers+text" if label else "markers",
            marker=dict(symbol="star", size=16, color=color, line=dict(color="#FFFFFF", width=1)),
            text=[f"{'Best' if m['hb'] else 'Lowest'}: {fmt_val(d.loc[idx, '_Y'], m)} ({d.loc[idx, 'SEASON_YEAR']})"] if label else None,
            textposition="top center", textfont=dict(color=color, size=11), hoverinfo="skip", showlegend=False,
            legendgroup=p, cliponaxis=False), **kw)
    if awards:
        for key, sym, sz, lab in (("champion_seasons", "triangle-up", 13, "NBA champion"), ("mvp_seasons", "circle-open", 20, "League MVP")):
            r = d[d["SEASON_YEAR"].isin(season_list(p, key))]
            if len(r):
                fill = "#FFFFFF" if sym == "triangle-up" else "rgba(0,0,0,0)"
                fig.add_trace(go.Scatter(
                    x=r[xcol], y=r["_Y"], mode="markers", showlegend=False, legendgroup=p,
                    marker=dict(symbol=sym, size=sz, color=fill, line=dict(color=color if sym == "triangle-up" else "#FFFFFF", width=2.5)),
                    customdata=r[["SEASON_YEAR"]].astype(object).values.tolist(),
                    hovertemplate=f"<b>{p}</b> · %{{customdata[0]}}<br>{lab} season<extra></extra>"), **kw)


def empty_msg():
    return HTML('<div class="warnbox">Select at least one player to see this view.</div>')


def fig_trend(mkey, players, xmode, smooth, awards=True):
    xcol, xtitle = xinfo(xmode)
    fig = go.Figure()
    for p in players:
        add_series(fig, p, mkey, xcol, smooth, awards=awards)
    style(fig, f"{mkey} by season", height=520, margin=dict(l=65, r=30, t=75, b=55))
    fig.update_xaxes(title=xtitle, dtick=1 if xmode == "Career year" else 2)
    fig.update_yaxes(title=mkey, tickformat=".0%" if METRICS[mkey]["kind"] == "pct" else None)
    fig.update_layout(hovermode="closest")
    return fig


# ---- fun head-to-head insights for the selected metric ------------------------------------------------
METRIC_VOICE = {   # metric -> (how to say it in a sentence, name of the belt)
    "Points per game": ("scoring", "Bucket Boss"),
    "True shooting % (TS%)": ("scoring efficiency", "Efficiency Wizard"),
    "Effective FG% (eFG%)": ("shot-making", "Shot Sniper"),
    "Field goal %": ("field-goal accuracy", "Accuracy Ace"),
    "3-point %": ("long-range accuracy", "Deep-Ball Dealer"),
    "Free throw %": ("free-throw touch", "Charity-Stripe Surgeon"),
    "Assists per game": ("dime-dropping", "Dime Dealer"),
    "Rebounds per game": ("glass-cleaning", "Board Boss"),
    "Steals per game": ("pickpocketing", "Pickpocket Prime"),
    "Blocks per game": ("shot-swatting", "Swat Team Captain"),
    "Turnovers per game": ("ball security", "Safe Hands Champion"),
    "3-pt attempts per game": ("three-point volume", "Long-Range Enthusiast"),
    "Free-throw attempts per game": ("free-throw trips", "Foul-Line Frequent Flyer"),
    "Games played (regular season)": ("availability", "Iron Man"),
    "Regular-season win %": ("winning", "Win Machine"),
}
assert set(METRIC_VOICE) == set(METRICS), "every metric needs a voice"


def _gap(v, m):
    v = abs(v)
    return f"{v * 100:.1f} pp" if m["kind"] == "pct" else f"{v:,.{m['dec']}f}"


def _delta(v, m):
    return f"{v * 100:+.1f} pp" if m["kind"] == "pct" else f"{v:+,.{m['dec']}f}"


def trend_insights(mkey, players):
    """Returns (headline_html, subline_html, rows); each row = (label, sub-label, winner-or-None, commentary_html)."""
    m = METRICS[mkey]
    noun, crown = METRIC_VOICE[mkey]
    col, hb = m["col"], m["hb"]
    var = len(mkey) % 3                                   # picks one of three phrasings, deterministically per metric
    fv = lambda v: fmt_val(v, m)
    sh = lambda p: f'<b style="color:{PLAYER_COLORS[p]}">{PLAYER_SHORT[p]}</b>'
    best_of = (lambda d: max(d, key=d.get)) if hb else (lambda d: min(d, key=d.get))
    order_of = lambda d: sorted(d, key=d.get, reverse=hb)

    S = {}
    for p in players:
        d = SEASON_BY_PLAYER[p]
        q = d if col == "GAMES_PLAYED_REGULAR_SEASON" else d[d["QUALIFIED"]]
        q = q.dropna(subset=[col]).reset_index(drop=True)
        if len(q):
            S[p] = q
    players = [p for p in players if p in S]
    multi = len(players) >= 2
    rows = []

    # 1 · best single season ---------------------------------------------------------------------------
    peak = {}
    for p in players:
        q = S[p]
        i = q[col].idxmax() if hb else q[col].idxmin()
        peak[p] = (float(q.loc[i, col]), q.loc[i, "SEASON_YEAR"])
    w = best_of({p: v[0] for p, v in peak.items()})
    if multi:
        r = order_of({p: v[0] for p, v in peak.items()})[1]
        gap = _gap(peak[w][0] - peak[r][0], m)
        if hb:
            t = [f"{sh(w)} went full video-game mode in {peak[w][1]}: <b>{fv(peak[w][0])}</b>. {sh(r)} peaked at {fv(peak[r][0])} and politely clapped.",
                 f"{peak[w][1]} was {sh(w)}'s victory lap: <b>{fv(peak[w][0])}</b>, {gap} clear of the next-best peak ({sh(r)}). Frame it.",
                 f"Highest {noun} ceiling: {sh(w)} (<b>{fv(peak[w][0])}</b>, {peak[w][1]}). {sh(r)} tops out {gap} lower. Bring a ladder."][var]
        else:
            t = [f"{sh(w)} guarded the rock like a family recipe in {peak[w][1]}: just <b>{fv(peak[w][0])}</b>. {sh(r)}'s tightest year was {fv(peak[r][0])}.",
                 f"{peak[w][1]} was {sh(w)}'s clean-sheet season: <b>{fv(peak[w][0])}</b>, {gap} stingier than {sh(r)}'s best. Butter-free fingers.",
                 f"Best {noun} ever: {sh(w)} (<b>{fv(peak[w][0])}</b>, {peak[w][1]}). {sh(r)} comes up {gap} short. Velcro gloves confirmed."][var]
    else:
        t = f"{sh(w)}'s best {noun} season: {peak[w][1]} at <b>{fv(peak[w][0])}</b>. Not bad. Not bad at all."
    rows.append(("Best single season", "the peak performance", w, t))

    # 2 · typical season ---------------------------------------------------------------------------------
    avg = {p: float(S[p][col].mean()) for p in players}
    order = order_of(avg)
    w = order[0]
    chain = " › ".join(f"{sh(p)} <b>{fv(avg[p])}</b>" for p in order)
    if multi:
        t = [f"Season in, season out: {chain}. The tale of the tape.",
             f"The long game: {chain}. Consistency pays, and {sh(w)} cashed the check.",
             f"Average-season showdown: {chain}. {sh(w)} wins the marathon, not just the sprint."][var]
    else:
        t = f"Typical season: <b>{fv(avg[w])}</b>. Respectable."
    rows.append(("Typical season", "average of qualifying seasons", w, t))

    # 3 · steadiness ---------------------------------------------------------------------------------------
    cv = {p: float(S[p][col].std(ddof=0) / S[p][col].mean() * 100) for p in players if len(S[p]) >= 3 and S[p][col].mean() != 0}
    if cv:
        steady = min(cv, key=cv.get)
        if len(cv) >= 2:
            wild = max(cv, key=cv.get)
            t = [f"{sh(steady)} is the metronome: only ±{cv[steady]:.1f}% swing between seasons. {sh(wild)} treated every year like a plot twist (±{cv[wild]:.1f}%).",
                 f"Steady Eddie award: {sh(steady)} (±{cv[steady]:.1f}%). Rollercoaster award: {sh(wild)} (±{cv[wild]:.1f}%). Hands up, screaming optional.",
                 f"{sh(steady)} sets the cruise control (±{cv[steady]:.1f}%). {sh(wild)} keeps the seatbelt sign on (±{cv[wild]:.1f}%)."][var]
        else:
            t = f"{sh(steady)} swings just ±{cv[steady]:.1f}% from season to season. Cruise control engaged."
        rows.append(("Steadiness", "season-to-season swing", steady if len(cv) >= 2 else None, t))

    # 4 · glow-up or fade ----------------------------------------------------------------------------------
    shift = {}
    for p in players:
        q = S[p]
        if len(q) >= 6:
            f0, l0 = float(q[col].head(3).mean()), float(q[col].tail(3).mean())
            if f0 != 0:
                shift[p] = (f0, l0, l0 / f0 - 1)
    if shift:
        score = {p: (v[2] if hb else -v[2]) for p, v in shift.items()}
        w = max(score, key=score.get)
        fw, lw, cw = shift[w]
        if len(shift) >= 2:
            lo = min(score, key=score.get)
            ff, lf, cf = shift[lo]
            lead = (f"{sh(w)} is aging like fine wine: {fv(fw)} → <b>{fv(lw)}</b> ({cw:+.0%})." if score[w] > 0
                    else f"Father Time got everybody, but {sh(w)} negotiated the best deal: {fv(fw)} → <b>{fv(lw)}</b> ({cw:+.0%}).")
            tail = (f" {sh(lo)} went {fv(ff)} → {fv(lf)} ({cf:+.0%}); Father Time sends regards." if score[lo] < 0
                    else f" {sh(lo)} nudged {fv(ff)} → {fv(lf)} ({cf:+.0%}). Slow and steady.")
            t = lead + tail
        else:
            t = f"{sh(w)}: first 3 seasons {fv(fw)}, last 3 seasons <b>{fv(lw)}</b> ({cw:+.0%}). " + ("Wine, not milk." if score[w] > 0 else "The odometer is showing.")
        rows.append(("Glow-up or fade", "first 3 seasons vs last 3", w if len(shift) >= 2 else None, t))

    if multi:
        # 5 · year-by-year bracket (same career year vs same career year) -------------------------------
        years = sorted(set().union(*[set(S[p]["PLAYER_SEASON_NUM"]) for p in players]))
        wins, n = {p: 0 for p in players}, 0
        for y in years:
            vals = {p: float(S[p].loc[S[p]["PLAYER_SEASON_NUM"] == y, col].iloc[0]) for p in players if (S[p]["PLAYER_SEASON_NUM"] == y).any()}
            if len(vals) >= 2:
                n += 1
                wins[best_of(vals)] += 1
        if n:
            order = sorted(players, key=lambda p: wins[p], reverse=True)
            tie = wins[order[0]] == wins[order[1]]
            tally = " · ".join(f"{sh(p)} <b>{wins[p]}</b>" for p in order)
            t = [f"Year 1 vs Year 1, Year 2 vs Year 2… {tally} out of {n} showdowns. Rookie to veteran, every round counted.",
                 f"Bracket check ({n} same-career-year matchups): {tally}. " + ("Dead heat at the top!" if tie else f"{sh(order[0])} owns the bracket."),
                 f"If careers were a tournament: {tally} across {n} rounds. " + ("Overtime needed." if tie else f"{sh(order[0])} advances.")][var]
            rows.append(("Year-by-year bracket", "who led in each career year", None if tie else order[0], t))

        # 6 · above-the-pack seasons ------------------------------------------------------------------------
        med = float(pd.concat([S[p][col] for p in players]).median())
        beat = {p: (S[p][col] > med) if hb else (S[p][col] < med) for p in players}
        share = {p: float(beat[p].mean()) for p in players}
        order = sorted(players, key=lambda p: share[p], reverse=True)
        hi, lo = order[0], order[-1]
        tie = share[hi] == share[order[1]]
        t = [f"{sh(hi)} beat the group's typical season {int(beat[hi].sum())} of {len(S[hi])} times (<b>{share[hi]:.0%}</b>). {sh(lo)}: {share[lo]:.0%}. Plenty of good years, just fewer 'wow' ones.",
             f"Above-the-pack seasons: " + " · ".join(f"{sh(p)} <b>{share[p]:.0%}</b>" for p in order) + ". Showing up is half the battle; showing out is the other half.",
             f"{sh(hi)} lived above the group's median year <b>{share[hi]:.0%}</b> of the time. {sh(lo)} was there {share[lo]:.0%}. Receipts attached."][var]
        rows.append(("Above-the-pack seasons", "share of seasons beating the group's median", None if tie else hi, t))

    # 7 · championship mode ----------------------------------------------------------------------------------
    tm = {}
    for p in players:
        q = S[p]
        is_t = q["SEASON_YEAR"].isin(season_list(p, "champion_seasons"))
        if is_t.any() and (~is_t).any():
            tm[p] = (float(q.loc[is_t, col].mean()), float(q.loc[~is_t, col].mean()))
    if tm:
        imp = {p: ((v[0] - v[1]) if hb else (v[1] - v[0])) for p, v in tm.items()}
        w = max(imp, key=imp.get)
        per = " · ".join(f"{sh(p)} {fv(v[0])} vs {fv(v[1])}" for p, v in tm.items())
        if imp[w] > 0:
            t = f"Title years vs the rest: {per}. {sh(w)} flips a switch when the trophy is on the line ({_delta(tm[w][0] - tm[w][1], m)})."
        else:
            t = f"Title years vs the rest: {per}. Nobody leveled up in championship years. Already at full power, apparently."
        rows.append(("Championship mode", "title seasons vs all other seasons", w if len(tm) >= 2 else None, t))

    # headline ----------------------------------------------------------------------------------------------------
    tally = {}
    for _, _, win, _ in rows:
        if win:
            tally[win] = tally.get(win, 0) + 1
    n_cat = sum(1 for r_ in rows if r_[2])
    if multi and tally:
        top = sorted(tally.items(), key=lambda kv: kv[1], reverse=True)
        score_line = " · ".join(f"{PLAYER_SHORT[p]} {c}" for p, c in top)
        if len(top) > 1 and top[0][1] == top[1][1]:
            head, subl = f"{noun.capitalize()}: too close to call", f"Category wins: {score_line}. The {crown} belt stays in the vault."
        else:
            head, subl = f"{crown} of the day: {sh(top[0][0])}", f"Won {top[0][1]} of {n_cat} categories · {score_line}"
    else:
        head, subl = (f"Solo showcase: {sh(players[0])}", f"Flying solo on {noun}. Select another player to start a rivalry.") if players else ("", "")
    return head, subl, rows


def insights_html(mkey, players):
    if not players:
        return '<div class="warnbox">Select at least one player to see the scoreboard.</div>'
    head, subl, rows = trend_insights(mkey, players)
    body = "".join(
        f'<tr><td class="ik"><b>{lab}</b><span class="s">{sub}</span>'
        + (f'<span class="chip" style="background:{PLAYER_COLORS[win]};color:{PLAYER_TEXT[win]}">{PLAYER_SHORT[win]}</span>' if win else '<span class="chip" style="background:#2A3550;color:#E8ECF4">Draw</span>')
        + f'</td><td class="iv">{txt}</td></tr>' for lab, sub, win, txt in rows)
    return (f'<div class="ins-wrap"><div class="ins-head"><div class="ins-kicker">The scoreboard · {mkey}</div>'
            f'<div class="ins-title">{head}</div><div class="ins-sub">{subl}</div></div>'
            f'<table class="itbl"><tbody>{body}</tbody></table>'
            f'<div class="ins-foot">Qualifying seasons only (30+ games). Computed live from the data; all commentary is in good fun.</div></div>')

In [ ]:
def build_trend_explorer():
    boxes = {p: W.Checkbox(value=True, description=p, indent=False, layout=W.Layout(width="165px")) for p in PLAYERS}
    for p, b in boxes.items():
        b.add_class(f"pbox-{PLAYER_SLUG[p]}")
    xmode = W.ToggleButtons(options=["Career year", "Calendar season"], value="Career year", description="X-axis", style={"description_width": "50px"})
    smooth = W.Checkbox(value=False, description="3-season rolling average", indent=False, layout=W.Layout(width="230px"))
    awards_cb = W.Checkbox(value=True, description="Mark title ▲ and MVP ○ seasons", indent=False, layout=W.Layout(width="290px"))
    metric_dd = W.Dropdown(options=list(METRICS), value="Points per game", description="Metric", style={"description_width": "55px"}, layout=W.Layout(width="380px"))
    chart_out = W.Output(layout=W.Layout(width="63%"))
    ins_html = W.HTML(layout=W.Layout(width="37%", margin="0 0 0 10px"))

    def players():
        return [p for p in PLAYERS if boxes[p].value]

    def render(_=None):
        ps = players()
        with chart_out:
            clear_output(wait=True)
            if not ps:
                display(empty_msg())
            else:
                show(fig_trend(metric_dd.value, ps, xmode.value, smooth.value, awards_cb.value))
                display(caption("★ = best season for the selected metric (lowest for turnovers). Hollow markers = seasons with fewer than 30 games; "
                                "gaps in a line = calendar seasons missing from the data. ▲ = NBA-title season, ○ = league-MVP season. Stats combine regular season and playoffs."))
        ins_html.value = insights_html(metric_dd.value, ps)

    for wdg in list(boxes.values()) + [xmode, smooth, awards_cb, metric_dd]:
        wdg.observe(render, names="value")
    root = W.VBox([
        W.HTML('<div class="sec">Controls</div>'),
        W.HBox([W.HTML("<b style='color:#8A96B0'>Players&nbsp;&nbsp;</b>")] + [boxes[p] for p in PLAYERS]),
        W.HBox([xmode, smooth, awards_cb]),
        metric_dd,
        W.HBox([chart_out, ins_html], layout=W.Layout(width="100%", align_items="flex-start")),
    ])
    root.add_class("goat-root")
    root.refs = dict(boxes=boxes, xmode=xmode, smooth=smooth, awards=awards_cb, metric=metric_dd, chart=chart_out, ins=ins_html, render=render)
    render()
    return root


TREND_UI = build_trend_explorer()
display(TREND_UI)

## Methodology & limitations
* **Sources.** GOAT Calculator: `game_logs_summary_career_overall.csv` and `player_accolades.csv`. Trend Explorer: `game_logs_summary_per_season.csv`. Per-season points are derived as `2·FGM + 3PM + FTM` and reconciled with the career `PTS_AVE`; stats combine regular season and playoffs.
* **Normalization.** Each row is scaled 0–100 against the best of the three players (`value ÷ best`; turnovers use `best ÷ value`). Ratio scaling compresses near-ties (e.g. free-throw %), so rows with small real gaps still look close.
* **Weighting.** Weights 1–10 become shares of the total weight; each player's total is the sum of normalized score × share, so a total of 100 would mean leading every row.
* **Accolades are external.** They are hand-compiled (5 Nov 2026) from NBA.com, ESPN and Wikipedia/Basketball-Reference-based lists, not from the game logs. Voting rules and eligibility change over time, and counts such as Finals MVPs depend on teammates and opponents.
* **Final Verdict.** The calculator has 15 sliders from 1 to 10, i.e. 10¹⁵ combinations, which cannot be enumerated one by one. Counts are exact on coarse grids (2¹⁵, 3¹⁵ and 4¹⁵ combinations, all evaluated), estimated from 20 million random combinations on the full scale (95% intervals shown), and a geometric test proves when a player cannot be Rank #1 under any setting. The result depends on which rows the calculator uses (for example 6 of the 15 are accolades) and on how they were counted.
* **No volume bonus.** Total points, total games and seasons played are not used in the calculator, so a longer career earns no automatic edge.
* **Insights are playful.** The commentary beside the Trend Explorer is generated from the data with fixed templates; it is meant to be fun, not definitive. Qualifying seasons only (30+ games).
* **Small sample.** Three players means every ranking is relative, and a GOAT score is a model of *your* priorities, not an objective truth.

# 3 · Final Verdict
**Who is Rank #1 across every way you could set the sliders?**
The calculator has 15 sliders and each runs from 1 to 10, so there are **10¹⁵ = 1,000,000,000,000,000** possible weight combinations. A player is Rank #1 in a combination when their Total Weighted Score is the highest. Checking a quadrillion combinations one by one is not realistic, so the verdict combines three methods:
1. **Exact counts on coarse grids:** every combination where each slider is set to one of a few levels (Low/High: 2¹⁵ = 32,768 · Low/Medium/High: 3¹⁵ = 14,348,907 · four levels: 4¹⁵ = 1,073,741,824). Every single combination is evaluated, none are sampled.
2. **A very large random sample of the full 1–10 scale:** 20 million random combinations, scaled up to the 10¹⁵ total, with a 95% confidence interval.
3. **An exact lock-out test:** a geometric check over *all* slider settings that proves when a player cannot be Rank #1 under any setting at all.
It uses the same normalized scores as the calculator (they do not depend on where your sliders currently sit), so it runs once and never changes when you drag a slider.

In [ ]:
import time

VERDICT_GRIDS = [("Low / High only", (1, 10)), ("Low / Medium / High", (1, 5, 10)), ("Four levels (1 · 4 · 7 · 10)", (1, 4, 7, 10))]
VERDICT_SAMPLES = 20_000_000          # random combinations drawn from the full 1-10 scale
VERDICT_SEED = 2026
N_ROWS = len(CALC_KEYS)
SCORE_M = NORM.loc[PLAYERS, CALC_KEYS].to_numpy().T          # 15 rows x 3 players: normalized scores (0-100)
ROW_NAME = {r[0]: r[2] for r in CALC_ROWS}
TOTAL_COMBOS = (WEIGHT_MAX - WEIGHT_MIN + 1) ** N_ROWS       # 10**15
EPS = 1e-9                                                   # score differences below this count as a tie


def exhaustive_counts(levels):
    """Evaluate EVERY combination of the given weight levels across all 15 rows (len(levels)**15 combinations).
    Meet-in-the-middle: sum the first 7 rows and the last 8 rows separately, then add every pair of partial sums."""
    lv = np.array(levels, float)
    D = np.stack([SCORE_M[:, 0] - SCORE_M[:, 1], SCORE_M[:, 0] - SCORE_M[:, 2]], axis=1)    # x = S_0 - S_1, y = S_0 - S_2 per row

    def half(rows):
        grid = np.stack(np.meshgrid(*[lv] * len(rows), indexing="ij"), -1).reshape(-1, len(rows))
        return grid @ D[rows]

    A, B = half(list(range(7))), half(list(range(7, N_ROWS)))
    Ax, Ay, Bx, By = A[:, 0], A[:, 1], B[:, 0], B[:, 1]
    step = max(1, 1_000_000 // len(B))
    wins = np.zeros(3, np.int64)
    for i in range(0, len(A), step):
        X = Ax[i:i + step, None] + Bx[None, :]
        Y = Ay[i:i + step, None] + By[None, :]
        wins[0] += np.count_nonzero((X > EPS) & (Y > EPS))                   # player 0 beats both rivals
        wins[1] += np.count_nonzero((X < -EPS) & (Y - X > EPS))              # player 1 beats player 0 and player 2
        wins[2] += np.count_nonzero((Y < -EPS) & (X - Y > EPS))              # player 2 beats player 0 and player 1
    total = len(levels) ** N_ROWS
    return dict(wins=wins, ties=int(total - wins.sum()), total=total)


def sampled_counts(n, seed=VERDICT_SEED, chunk=500_000):
    """Draw n random slider settings (each of the 15 weights uniform on WEIGHT_MIN..WEIGHT_MAX)."""
    rng = np.random.default_rng(seed)
    wins, ties, done = np.zeros(3, np.int64), 0, 0
    rank_counts, wsum = np.zeros((3, 3), np.int64), np.zeros((N_ROWS, 3))
    while done < n:
        c = min(chunk, n - done)
        W = rng.integers(WEIGHT_MIN, WEIGHT_MAX + 1, size=(c, N_ROWS))
        S = W @ SCORE_M
        order = np.argsort(-S, axis=1)
        srt = np.take_along_axis(S, order, axis=1)
        uniq = (srt[:, 0] - srt[:, 1]) > EPS
        ranks = np.empty_like(order)
        np.put_along_axis(ranks, order, np.broadcast_to(np.arange(3), order.shape), axis=1)
        onehot = np.zeros((c, 3))
        onehot[np.arange(c), order[:, 0]] = uniq
        wins += onehot.sum(axis=0).astype(np.int64)
        wsum += W.T @ onehot                                              # sum of the sliders over each player's winning settings
        for p in range(3):
            rank_counts[p] += np.bincount(ranks[:, p], minlength=3)
        ties += int((~uniq).sum())
        done += c
    return dict(wins=wins, ties=ties, n=n, rank_counts=rank_counts, avg_w=wsum / np.maximum(wins, 1))


def winning_profiles(levels=(1, 5, 10), chunk=400_000):
    """Exact: for every combination of the given levels, add up the slider settings of whoever is Rank #1, then average them per player."""
    lv, L = np.array(levels, float), len(levels)
    total = L ** N_ROWS
    wsum, wins = np.zeros((N_ROWS, 3)), np.zeros(3, np.int64)
    for start in range(0, total, chunk):
        idx = np.arange(start, min(total, start + chunk))
        W = np.empty((len(idx), N_ROWS))
        for k in range(N_ROWS):
            W[:, k] = lv[(idx // L ** k) % L]
        S = W @ SCORE_M
        srt = np.sort(S, axis=1)
        uniq = (srt[:, 2] - srt[:, 1]) > EPS
        onehot = np.zeros((len(idx), 3))
        onehot[np.arange(len(idx)), S.argmax(axis=1)] = uniq
        wins += onehot.sum(axis=0).astype(np.int64)
        wsum += W.T @ onehot
    return dict(levels=levels, wins=wins, total=total, avg_w=wsum / np.maximum(wins, 1), base=float(np.mean(levels)))


def wilson(k, n, z=1.96):
    """95% confidence interval for a proportion k/n."""
    if n == 0:
        return 0.0, 0.0
    p, d = k / n, 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return max(0.0, centre - half), min(1.0, centre + half)


def best_case_margin(p):
    """Exact: the largest value of min(S_p - S_q, S_p - S_r) over ALL weights in [WEIGHT_MIN, WEIGHT_MAX]^15 (S = weighted sum of scores).
    The set of reachable (S_p - S_q, S_p - S_r) pairs is a zonotope (a convex polygon), so the maximum sits on a vertex or where an edge crosses the diagonal.
    A value <= 0 proves that player can never be Rank #1, whatever the sliders say."""
    q, r = [i for i in range(3) if i != p]
    G = np.stack([SCORE_M[:, p] - SCORE_M[:, q], SCORE_M[:, p] - SCORE_M[:, r]], axis=1)
    lo, hi = float(WEIGHT_MIN), float(WEIGHT_MAX)
    ang = np.arctan2(G[:, 1], G[:, 0])
    crit = np.unique(np.round(np.mod(np.concatenate([ang + np.pi / 2, ang - np.pi / 2]), 2 * np.pi), 12))
    mids = (crit + np.append(crit[1:], crit[0] + 2 * np.pi)) / 2
    verts = []
    for th in mids:
        n = np.array([np.cos(th), np.sin(th)])
        verts.append(lo * G.sum(axis=0) + (hi - lo) * G[G @ n > 0].sum(axis=0))
    verts = np.array(verts)
    best = np.min(verts, axis=1).max()
    for a, b in zip(verts, np.roll(verts, -1, axis=0)):
        da, db = a[0] - a[1], b[0] - b[1]
        if da * db < 0:
            pt = a + (da / (da - db)) * (b - a)
            best = max(best, pt[0])
    return float(best)


# ---- run everything -------------------------------------------------------------------------------------------------------------
t0 = time.time()
GRID_RESULTS = []
for label, levels in VERDICT_GRIDS:
    res = exhaustive_counts(levels)
    res.update(label=label, levels=levels)
    GRID_RESULTS.append(res)
    print(f"exhaustive · {label}: {res['total']:,} combinations checked ({time.time() - t0:.1f}s)")
MC = sampled_counts(VERDICT_SAMPLES)
print(f"random sample of the full {WEIGHT_MIN}-{WEIGHT_MAX} scale: {VERDICT_SAMPLES:,} combinations ({time.time() - t0:.1f}s)")
PROFILE = winning_profiles((1, 5, 10))
print(f"winning-setting profile: all {PROFILE['total']:,} Low/Medium/High combinations enumerated ({time.time() - t0:.1f}s)")
MARGIN = {p: best_case_margin(i) for i, p in enumerate(PLAYERS)}
LOCKED_OUT = {p: MARGIN[p] <= EPS for p in PLAYERS}
print("exact lock-out test:", {PLAYER_SHORT[p]: ("cannot be #1 under any setting" if LOCKED_OUT[p] else "can be #1") for p in PLAYERS})
sanity = [int(r["wins"].sum() + r["ties"]) == r["total"] for r in GRID_RESULTS] + [int(MC["wins"].sum() + MC["ties"]) == MC["n"]]
assert all(sanity), "counts do not add up"
assert all(MC["wins"][i] == 0 for i, p in enumerate(PLAYERS) if LOCKED_OUT[p]), "a locked-out player won a sampled combination"
assert all(r["wins"][i] == 0 for r in GRID_RESULTS for i, p in enumerate(PLAYERS) if LOCKED_OUT[p]), "a locked-out player won an exhaustive combination"


# ---- present ------------------------------------------------------------------------------------------------------------------------
def _pct(x):
    """Percentage with enough digits to show the smaller of (share, 100 - share) to two significant figures."""
    p = x * 100
    q = min(p, 100 - p)
    if p <= 0:
        return "0%"
    if q <= 1e-9:
        return "100%"
    return f"{p:.{2 if q >= 1 else min(7, int(np.ceil(-np.log10(q))) + 1)}f}%"


def _rare(x):
    return f" (≈ {x * 1e6:,.0f} in a million)" if 0 < x < 1e-3 else ""


def _count_est(share):
    """Readable estimated count out of 10**15 combinations, with enough digits to tell 999.9996 trillion apart from 1,000 trillion."""
    val = share * TOTAL_COMBOS
    if val <= 0:
        return "0 in the sample"
    for unit, name in ((1e12, "trillion"), (1e9, "billion"), (1e6, "million")):
        if val >= unit:
            dec = 4 if share > 0.99 else 2
            return f"{val / unit:,.{dec}f} {name}"
    return f"{val:,.0f}"


def _cell(p, count_html, sub, best):
    c = PLAYER_COLORS[p]
    ring = f"box-shadow:inset 0 0 0 2px {c};background:{rgba(c, 0.16)};font-weight:700;" if best else ""
    return f'<td style="{ring}">{count_html}<span class="sub">{sub}</span></td>'


def verdict_table_html():
    head = "".join(f'<th style="background:{PLAYER_COLORS[p]};color:{PLAYER_TEXT[p]}">{p}<span class="sub" style="color:inherit;opacity:.85">Rank #1 count</span></th>' for p in PLAYERS)
    body = ""
    for r in GRID_RESULTS:
        best = int(np.argmax(r["wins"]))
        cells = "".join(_cell(p, f"{int(r['wins'][i]):,}", _pct(r["wins"][i] / r["total"]) if r["wins"][i] else ("impossible" if LOCKED_OUT[p] else "0"), i == best)
                        for i, p in enumerate(PLAYERS))
        body += (f'<tr><td class="l"><b>{r["label"]}</b><span class="sub">every combination checked · exact</span></td><td>{r["total"]:,}</td>{cells}</tr>')
    n = MC["n"]
    best = int(np.argmax(MC["wins"]))
    cells = ""
    for i, p in enumerate(PLAYERS):
        k = int(MC["wins"][i])
        if LOCKED_OUT[p]:
            cells += _cell(p, "0", "impossible (proved)", False)
        else:
            lo, hi = wilson(k, n)
            cells += _cell(p, f"≈ {_count_est(k / n)}", f"{_pct(k / n)} · 95% CI {_pct(lo)} – {_pct(hi)}", i == best)
    body += (f'<tr><td class="l"><b>Full {WEIGHT_MIN}–{WEIGHT_MAX} scale</b><span class="sub">{n:,} random combinations · estimated</span></td>'
             f'<td>10<sup>15</sup><span class="sub">1 quadrillion</span></td>{cells}</tr>')
    return (f'<table class="fvt"><thead><tr><th class="l">Scenario</th><th>Combinations</th>{head}</tr></thead><tbody>{body}</tbody></table>')


def rank_table_html():
    rows = ""
    for i, p in enumerate(PLAYERS):
        rc = MC["rank_counts"][i] / MC["n"]
        cells = "".join(f'<td style="{"font-weight:700;color:" + PLAYER_COLORS[p] if j == int(np.argmax(rc)) else ""}">{_pct(rc[j])}</td>' for j in range(3))
        rows += f'<tr><td class="l"><b style="color:{PLAYER_COLORS[p]}">{p}</b></td>{cells}</tr>'
    return (f'<table class="fvt" style="max-width:520px"><thead><tr><th class="l">Finishing position<span class="sub">share of {MC["n"]:,} random settings</span></th>'
            f'<th>Rank #1</th><th>Rank #2</th><th>Rank #3</th></tr></thead><tbody>{rows}</tbody></table>')


def winning_profile_html():
    """Average slider setting inside the combinations each player wins, against the all-combinations average (exact, Low/Medium/High grid)."""
    base, avg = PROFILE["base"], PROFILE["avg_w"]
    contenders = [i for i in range(3) if PROFILE["wins"][i] >= 100]
    if not contenders:
        return ""
    head = "".join(f'<th style="background:{PLAYER_COLORS[PLAYERS[i]]};color:{PLAYER_TEXT[PLAYERS[i]]}">{PLAYERS[i]}'
                   f'<span class="sub" style="color:inherit;opacity:.85">{int(PROFILE["wins"][i]):,} winning settings</span></th>' for i in contenders)
    body, notes = "", []
    for k, key in enumerate(CALC_KEYS):
        cells = ""
        for i in contenders:
            dev = avg[k, i] - base
            c = PLAYER_COLORS[PLAYERS[i]]
            tint = f"background:{rgba(c, min(0.5, abs(dev) / 4.0))};font-weight:700;" if dev > 0.25 else ""
            cells += f'<td style="{tint}">{avg[k, i]:.1f}<span class="sub">{(dev if abs(dev) >= 0.05 else 0.0):+.1f}</span></td>'
        body += f'<tr><td class="l">{ROW_NAME[key]}</td>{cells}</tr>'
    for i in contenders:
        dev = avg[:, i] - base
        name = f'<b style="color:{PLAYER_COLORS[PLAYERS[i]]}">{PLAYER_SHORT[PLAYERS[i]]}</b>'
        if np.abs(dev).max() < 0.5:
            notes.append(f"{name} wins {_pct(PROFILE['wins'][i] / PROFILE['total'])} of these combinations, so his winning settings look like everyone's "
                         f"(no slider's average differs from {base:.1f} by more than {np.abs(dev).max():.2f}). Almost any setting crowns him.")
        else:
            top = [f"{ROW_NAME[CALC_KEYS[k]]} ({avg[k, i]:.1f})" for k in np.argsort(-dev)[:3]]
            low = [f"{ROW_NAME[CALC_KEYS[k]]} ({avg[k, i]:.1f})" for k in np.argsort(dev)[:2]]
            notes.append(f'{name} is Rank #1 only when the sliders lean hard toward {", ".join(top)} and away from {", ".join(low)}.')
    return (f'<table class="fvt tight"><thead><tr><th class="l">Average slider setting in the combinations each player wins'
            f'<span class="sub">exact, over all {PROFILE["total"]:,} Low/Medium/High combinations · all-combinations average = {base:.1f} · shaded = leaning above it</span></th>{head}</tr></thead>'
            f'<tbody>{body}</tbody></table><div class="cap" style="margin-top:8px">{" ".join(notes)}</div>')


def verdict_banner_html():
    n = MC["n"]
    share = MC["wins"] / n
    order = list(np.argsort(-share))
    w = PLAYERS[order[0]]
    parts = []
    for i in order[1:]:
        p = PLAYERS[i]
        parts.append(f'{PLAYER_SHORT[p]}: ' + ("never (proved: no slider setting makes this possible)" if LOCKED_OUT[p] else f"{_pct(share[i])}{_rare(share[i])}"))
    return (f'<div class="ins-wrap"><div class="ins-head"><div class="ins-kicker">Final verdict · {TOTAL_COMBOS:,} possible slider combinations</div>'
            f'<div class="ins-title"><span style="color:{PLAYER_COLORS[w]}">{w}</span> is Rank #1 in {_pct(share[order[0]])} of them</div>'
            f'<div class="ins-sub">{" · ".join(parts)}</div></div>'
            f'<div class="ins-foot" style="border-top:none">Rank #1 = highest Total Weighted Score. The verdict depends on which 15 rows the calculator uses and how accolades were counted, '
            f'not just on the sliders (see the methodology notes above).</div></div>')


def fig_verdict():
    labels = [r["label"] for r in GRID_RESULTS] + [f"Full {WEIGHT_MIN}–{WEIGHT_MAX} scale (estimated)"]
    shares = np.array([r["wins"] / r["total"] for r in GRID_RESULTS] + [MC["wins"] / MC["n"]]) * 100
    fig = go.Figure()
    for i, p in enumerate(PLAYERS):
        fig.add_trace(go.Bar(y=labels, x=shares[:, i], orientation="h", name=p, marker_color=PLAYER_COLORS[p],
                             text=[f"{v:.2f}%" if v >= 4 else "" for v in shares[:, i]], textposition="inside", insidetextanchor="middle",
                             textfont=dict(color=PLAYER_TEXT[p], size=13),
                             hovertemplate=f"<b>{p}</b><br>%{{y}}<br>Rank #1 in %{{x:.4f}}% of combinations<extra></extra>"))
    style(fig, "Share of weight combinations in which each player is Rank #1", height=330, margin=dict(l=210, r=30, t=80, b=50))
    fig.update_layout(barmode="stack")
    fig.update_xaxes(title="% of combinations", range=[0, 100])
    fig.update_yaxes(autorange="reversed")
    return fig


display(HTML('<div class="goat-root" style="padding:14px">' + verdict_banner_html() + '<div class="sec">Rank #1 counts</div>' + verdict_table_html() + '</div>'))
show(fig_verdict())
display(caption("Bars show the share of combinations in which each player is Rank #1. A slice that is too thin to see (or to label) is still listed in the table above."))
display(HTML('<div class="goat-root" style="padding:14px"><div class="sec">Where each player finishes</div>' + rank_table_html()
             + '<div class="sec" style="margin-top:18px">What a winning slider setting looks like</div>' + winning_profile_html() + '</div>'))